# 📓 Semana 16 · Dia 3 — App RAG com UI de chat

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Assoc (deploy) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | App de chat RAG publicado |

---


## 📖 Teoria — Chat RAG em produção

O app de chat precisa de: histórico de mensagens, streaming (ou resposta rápida), **citações de fonte** e **feedback** (👍/👎) — o feedback alimenta o golden set.


### 💻 Na prática — App de chat

Streamlit com histórico e citações.


In [ ]:
# app.py — chat RAG
import streamlit as st
from databricks.vector_search.client import VectorSearchClient
from langchain_community.chat_models import ChatDatabricks

st.set_page_config(page_title="RAG Produtos", layout="wide")
st.title("🔍 Assistente de Produtos (RAG)")

vsc = VectorSearchClient()
idx = vsc.get_index("workspace.prata.produtos_rag_index")
llm = ChatDatabricks(endpoint="databricks-llama-3-1-70b", temperature=0.1)

if "historico" not in st.session_state:
    st.session_state.historico = []

pergunta = st.chat_input("Pergunte sobre os produtos...")
if pergunta:
    resultados = idx.similarity_search(query_text=pergunta, columns=["StockCode", "texto"], num_results=3)
    contexto = "\n".join(r[1] for r in resultados["result"]["data_array"])
    resposta = llm.invoke(f"Contexto:\n{contexto}\n\nPergunta: {pergunta}").content
    st.session_state.historico.append((pergunta, resposta, resultados))

for pergunta, resposta, resultados in st.session_state.historico:
    st.chat_message("user").write(pergunta)
    st.chat_message("assistant").write(resposta)
    with st.expander("Fontes"):
        for r in resultados["result"]["data_array"]:
            st.write(f"• {r[0]}: {r[1][:100]}")
print("App de chat RAG com fontes pronto.")

### 💻 Na prática — Feedback

Adicione botões 👍/👎 que gravam o feedback numa tabela.


In [ ]:
# Feedback gravado em Delta
import streamlit as st
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
def salvar_feedback(pergunta, resposta, nota):
    spark.createDataFrame([(pergunta, resposta, nota)], ["pergunta", "resposta", "nota"])\
        .write.mode("append").saveAsTable("workspace.audit.feedback_rag")
if st.button("👍 Útil"):
    salvar_feedback(pergunta, resposta, 1)
if st.button("👎 Não útil"):
    salvar_feedback(pergunta, resposta, 0)
print("Feedback gravado em workspace.audit.feedback_rag.")

> 🎯 **Dica de prova**: GenAI/portfólio: chat com citações + feedback é o padrão de produto RAG. O feedback alimenta o golden set e a melhoria contínua.


## 🎯 Exercícios de fixação

**1.** Publique o chat RAG e teste com 5 perguntas.

**2.** O que o feedback 👍/👎 permite melhorar?

**3.** Como as citações aumentam a confiança do usuário?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Teste

Verifique se as respostas usam o contexto e se as fontes correspondem.

**2.** Feedback

Identifica perguntas que falham → vira golden set → avaliação → melhoria do retrieval.

**3.** Citações

O usuário confere a fonte — transparência que reduz 'achismo' e aumenta adoção.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*